# Preprocess QC — main 151 + LOO 78 ecotypes

Sanity checks on the trim+dedup outputs:
1. **Presence** — does every expected ecotype have its dedup output (PE→2 files, SE→1)?
2. **Clumpify dup rate** — parsed from preprocess SLURM logs (`Reads In/Out`, `Duplicates Found`).
3. **Per-file integrity + read counts** — `gzip -t` + zcat-count via `qc_one.sh` SLURM array.
4. **R1/R2 parity, total bases, coverage estimate, read length** — derived from (3).
5. **ENA-vs-post comparison** — ENA-reported `read_count` vs post-trim `Reads In` and post-dedup `Reads Out`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUT = Path('../output')
PANELS = ['main', 'loo']
PANEL_COLORS = {'main': '#1f4e79', 'loo': '#d8b365'}

presence  = {p: pd.read_csv(OUT/f'presence_{p}.tsv',  sep='\t') for p in PANELS}
dup_rates = {p: pd.read_csv(OUT/f'dup_rates_{p}.tsv', sep='\t') for p in PANELS}

# ENA manifests carry input read_count + base_count which we'll cross-check
manifests = {
    'main': pd.read_csv('../../pangenie_genotyping/data/ena_manifest.tsv', sep='\t').rename(columns={'ecotype_id':'ecotype'}),
    'loo':  pd.read_csv('../../pangenie_genotyping/data/loo_ena_manifest.tsv', sep='\t').rename(columns={'ecotype_id':'ecotype'}),
}
for p in PANELS:
    manifests[p]['ecotype'] = manifests[p]['ecotype'].astype(str)
    presence[p]['ecotype']  = presence[p]['ecotype'].astype(str)
    dup_rates[p]['ecotype'] = dup_rates[p]['ecotype'].astype(str)

for p, df in presence.items():
    print(f'== {p} ==', dict(df.status.value_counts()))

## 1. Presence audit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, p in zip(axes, PANELS):
    s = presence[p]['status'].value_counts()
    ax.bar(s.index, s.values, color=['#2ca02c' if x=='OK' else '#d62728' for x in s.index])
    for i, v in enumerate(s.values):
        ax.text(i, v, str(v), ha='center', va='bottom', fontsize=11)
    ax.set_title(f'{p}: {len(presence[p])} ecotypes')
    ax.set_ylabel('# ecotypes')
plt.tight_layout(); plt.show()

# Show any non-OK rows
for p in PANELS:
    bad = presence[p][presence[p].status != 'OK']
    if len(bad):
        print(f'\n== {p} non-OK ==')
        display(bad)

## 2. Clumpify duplicate rates

Distribution of `Duplicates Found` percentages across samples. Healthy short-read libraries are typically 0–30%; >50% is suspicious.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, p in zip(axes, PANELS):
    df = dup_rates[p]
    ax.hist(df.dup_pct, bins=30, color=PANEL_COLORS[p], edgecolor='k', alpha=0.85)
    ax.axvline(df.dup_pct.median(), color='k', ls='--', lw=1, label=f'median {df.dup_pct.median():.1f}%')
    ax.axvline(50, color='r', ls=':', lw=1, label='alert >50%')
    ax.set_xlabel('Clumpify duplicate %')
    ax.set_ylabel('# ecotypes')
    ax.set_title(f'{p}  (n={len(df)})  mean {df.dup_pct.mean():.2f}%, max {df.dup_pct.max():.1f}%')
    ax.legend()
plt.tight_layout(); plt.show()

# Top 10 by dup rate per panel
for p in PANELS:
    print(f'\n== {p}: top 10 by dup_pct ==')
    print(dup_rates[p].nlargest(10, 'dup_pct')[['ecotype','reads_in','reads_out','dup_pct']].to_string(index=False))

## 3. Read retention (Reads_Out / Reads_In)

Trim drops low-quality and adapter-readthrough reads; dedup drops PCR duplicates. Combined retention = `Reads Out / Reads In`. Per xwu's lighter trim params (no SLIDINGWINDOW), retention should be high — typically 80–100%.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, p in zip(axes, PANELS):
    df = dup_rates[p].copy()
    df['retain_pct'] = df.reads_out / df.reads_in * 100
    ax.hist(df.retain_pct, bins=30, color=PANEL_COLORS[p], edgecolor='k', alpha=0.85)
    ax.axvline(df.retain_pct.median(), color='k', ls='--', lw=1, label=f'median {df.retain_pct.median():.1f}%')
    ax.axvline(50, color='r', ls=':', lw=1, label='alert <50%')
    ax.set_xlabel('reads retained (post-Clumpify / post-Trimmomatic) %')
    ax.set_title(f'{p}  (n={len(df)})  median {df.retain_pct.median():.1f}%')
    ax.legend()
plt.tight_layout(); plt.show()

## 4. ENA input → preprocess delta (input shrinkage end-to-end)

`ena.read_count` is what was deposited; `Reads In` is what Trimmomatic saw; `Reads Out` is post-dedup. Material gaps between ENA and Trimmomatic-In would suggest a download or preprocess-input issue.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, p in zip(axes, PANELS):
    df = dup_rates[p].merge(manifests[p][['ecotype','read_count']], on='ecotype', how='left')
    df['ena_to_trim_pct']  = df.reads_in  / df.read_count * 100
    df['ena_to_dedup_pct'] = df.reads_out / df.read_count * 100
    ax.hist([df.ena_to_trim_pct, df.ena_to_dedup_pct], bins=25,
            label=['ENA→Trim In','ENA→Dedup Out'], color=['#7f8da6','#1f4e79'])
    ax.set_xlabel('% of ENA read_count')
    ax.set_title(f'{p}: ENA -> preprocess pipeline')
    ax.legend()
plt.tight_layout(); plt.show()

for p in PANELS:
    df = dup_rates[p].merge(manifests[p][['ecotype','read_count','library_layout']], on='ecotype', how='left')
    df['ena_to_trim_pct']  = df.reads_in  / df.read_count * 100
    suspect = df[(df.ena_to_trim_pct < 95) | (df.ena_to_trim_pct > 110)]
    if len(suspect):
        print(f'\n== {p}: ena_to_trim outside 95-110% (potential issue) ==')
        print(suspect[['ecotype','read_count','reads_in','reads_out','ena_to_trim_pct','library_layout']].to_string(index=False))

## 5. Per-file QC results (gzip integrity, R1/R2 parity, coverage, read length)

Cell below loads `qc_results/` per-task TSVs (one per ecotype, written by `qc_one.sh` SLURM array). Run the array via:

```bash
PANEL=main sbatch --array=1-150%16 preprocess_qc/scripts/qc_one.sh
PANEL=loo  sbatch --array=1-52%16  preprocess_qc/scripts/qc_one.sh
```

After it finishes:

In [ ]:
qc_dir = OUT/'qc_results'
if not qc_dir.exists() or not any(qc_dir.iterdir()):
    print('qc_results/ is empty — submit the qc_one.sh array first.')
else:
    qc_all = pd.concat([pd.read_csv(p, sep='\t') for p in qc_dir.glob('*.tsv')], ignore_index=True)
    print(f'Loaded {len(qc_all)} QC rows')
    print('\ngzip integrity counts:')
    print(qc_all.groupby('panel')[['gz_r1','gz_r2']].apply(lambda d: pd.Series({'gz_r1_BAD':(d.gz_r1=='BAD').sum(),'gz_r2_BAD':(d.gz_r2=='BAD').sum()})))
    print('\nPE pair parity counts:')
    pe = qc_all[qc_all.layout=='PE']
    print(pe.groupby('panel')['pair_ok'].value_counts())
    print('\nCoverage estimates:')
    print(qc_all.groupby('panel')['cov_est'].describe())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, p in zip(axes, PANELS):
        sub = qc_all[qc_all.panel==p]
        ax.hist(sub.cov_est, bins=30, color=PANEL_COLORS[p], edgecolor='k')
        ax.set_title(f'{p}: estimated coverage (×) vs TAIR10 chr-only ~119.7Mb')
        ax.set_xlabel('coverage ×')
    plt.tight_layout(); plt.show()

## 6. Headline summary

In [ ]:
for p in PANELS:
    pres = presence[p]
    dup = dup_rates[p]
    print(f'== {p} ==')
    print(f'  ecotypes:        {len(pres)}')
    print(f'  presence OK:     {(pres.status=="OK").sum()}/{len(pres)}')
    print(f'  dup rate:        median {dup.dup_pct.median():.1f}%  mean {dup.dup_pct.mean():.2f}%  max {dup.dup_pct.max():.1f}%')
    retain = (dup.reads_out/dup.reads_in*100)
    print(f'  retention:       median {retain.median():.1f}%  min {retain.min():.1f}%')
    print()